# Initialize Data Assimilation with z-Only Observations

This notebook creates a new DA-system object for the z-only observation experiment. The dynamical state remains $(x,y,z)$, but the observation space has dimension one:

\[
y_k = H_z x_k + \varepsilon_k,\qquad
H_z=\begin{bmatrix}0&0&1\end{bmatrix},\qquad
R=[2].
\]

All ensemble, timing, model-state, and error settings are kept equal to the validated full-observation initialization. The output is written to a separate directory and does not replace `x_analysis_init.pkl`.


In [ ]:
from pathlib import Path

import numpy as np

from class_da_sys import da_system
from class_obs import obs_da
from class_state_vec import state_vector
from project_paths import ANALYSIS_DATA_DIR, NATURE_DATA_DIR, OBSERVATIONS_DATA_DIR

OBSERVATION_LABEL = 'z_only_obs'
OBSERVED_INDICES = (2,)

OBSERVATION_FILE = (
    OBSERVATIONS_DATA_DIR.parent / OBSERVATION_LABEL / 'data' / 'y_obs_z_only.pkl'
)
OUTPUT_ROOT = ANALYSIS_DATA_DIR.parent / OBSERVATION_LABEL
DATA_DIR = OUTPUT_ROOT / 'data'
FIGURES_DIR = OUTPUT_ROOT / 'figures'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = DATA_DIR / 'x_analysis_init_z_only_obs.pkl'

print('Observation input:', OBSERVATION_FILE)
print('DA-system output:', OUTPUT_FILE)


In [ ]:
sv = state_vector().load(NATURE_DATA_DIR / 'lorenz63_run.pkl')
obs = obs_da().load(OBSERVATION_FILE)

x_nature = np.asarray(sv.getTrajectory(), dtype=float)
y_obs = np.asarray(obs.getVal(), dtype=float)
y_pos = np.asarray(obs.getPos(), dtype=int)
xdim, maxit = x_nature.shape
ydim = y_obs.shape[0]

assert xdim == 3
assert ydim == 1
assert y_obs.shape[1] == maxit
assert np.all(y_pos == 2)

das = da_system()
das.setStateVector(sv)
das.setObsData(obs)
das.xdim = xdim
das.ydim = ydim
das.x0 = x_nature[:, 0]

das.edim = 3
das.ens_bias_init = 0
das.ens_sigma_init = np.sqrt(2)

das.setB(np.zeros((xdim, xdim)))
das.setR(2.0 * np.eye(ydim))
das.setH(np.array([[0.0, 0.0, 1.0]]))

t_nature = np.asarray(sv.getTimes())
das.acyc_step = 8
das.dtau = t_nature[das.acyc_step] - t_nature[0]
das.fcst_step = das.acyc_step
das.fcst_dt = das.dtau / das.fcst_step
das.dt = t_nature[1] - t_nature[0]
das.maxit = maxit
das.setMethod('EnKF')

das.observed_indices = list(OBSERVED_INDICES)
das.observation_label = OBSERVATION_LABEL
das.source_observation_file = str(OBSERVATION_FILE)
das.save(OUTPUT_FILE)


## Configuration Checks


In [ ]:
reloaded = da_system().load(OUTPUT_FILE)

expected_H = np.array([[0.0, 0.0, 1.0]])
assert reloaded.xdim == 3
assert reloaded.ydim == 1
assert reloaded.edim == 3
assert np.array_equal(np.asarray(reloaded.H), expected_H)
assert np.array_equal(np.asarray(reloaded.R), np.array([[2.0]]))
assert reloaded.acyc_step == 8
assert reloaded.getObsData().getVal().shape == (1, maxit)

print('State dimension:', reloaded.xdim)
print('Observation dimension:', reloaded.ydim)
print('Observed state indices:', reloaded.observed_indices)
print('H =\n', np.asarray(reloaded.H))
print('R =\n', np.asarray(reloaded.R))
print('Ensemble size:', reloaded.edim)
print('Analysis interval:', reloaded.acyc_step, 'model steps')
print('Saved and reloaded successfully:', OUTPUT_FILE)
